# Visualizing Transshipment Events using Plotly Dash and Folium

# Identifying Patterns of Transshipment Behavior That May Suggest Illegal Fishing
**Course:** CS 231 – Introduction to Data Science  
**Group:** Dhruv Patel, Franklin Collazo, Nufsat Tasmin  
**Date:** April 20 2026  

## **Project Summary:**
This project investigates global vessel behavior to identify patterns suggestive of illegal, unreported, and unregulated (IUU) fishing. We focus specifically on transshipment—the meeting of vessels at sea to transfer cargo. While transshipment is a standard maritime practice, it is often used as a cover for transferring illegally caught fish or avoiding port inspections.

**Using datasets from Global Fishing Watch, we analyze:**

 - Encounter Events: Instances where two vessels meet at sea.

- Loitering Events: Patterns where vessels hover in specific areas for extended durations.

- Vessel Registry: Characteristics of vessels capable of transshipment.

**Research Question:** Are there specific patterns in transshipment behavior (duration, location, and speed) that suggest potential illegal fishing activity?

**Methodology:**
As this is an unsupervised learning problem, we utilize:

 - Exploratory Data Analysis (EDA): To visualize geographic hotspots and behavior distributions.

- Principal Component Analysis (PCA): To reduce dimensionality and identify the most significant behavioral features.

- K-Means Clustering: To group vessel activities into distinct behavioral profiles for further ethical and environmental scrutiny.

**Ethical note:** Russia dominates the data; results are exploratory, not conclusive.

## 1. Import Libraries

Import the essential libraries for:
- **pandas** – data manipulation and analysis
- **plotly.express** – interactive visualizations (scatter maps, histograms)
- **folium** – interactive maps for geospatial plotting
- **os** – file path operations
- **dash** components – optional interactive dashboard elements (used for exploration; note the final notebook will primarily use static plots for reproducibility)

In [ ]:
import pandas as pd
import plotly.express as px
import folium
import os
import numpy as np

from dash import Dash, html, dcc, callback, Output, Input

from folium.plugins import MarkerCluster

from tqdm import tqdm

## 3. Load and Inspect the Three Core Datasets

Load three CSV files provided by Global Fishing Watch:
- **Encounter events** (11,681 rows × 9 columns): potential at‑sea meetings between a fishing vessel and a transshipment‑capable vessel.
- **Loitering events** (46,569 rows × 9 columns): periods when a transshipment vessel lingers in a small area, which may indicate waiting for a fishing vessel.
- **Transshipment vessels** (1,124 rows × 7 columns): metadata (MMSI, ship name, flag, IMO, first/last AIS timestamp) for vessels known to be capable of transshipment.

Print the shape (rows, columns) and the first row of each dataset to verify successful loading and understand the available fields.

In [ ]:
# CWD is the notebooks directory, as that is where Jupyter will run its kernel in.
TRANSSHIPMENT_DIR_FROM_CWD = "../data/raw/transshipment"
ENCOUNTER_EVENTS_FILENAME = "encounter-events-v20170717.csv"
LOITERING_EVENTS_FILENAME = "loitering-events-v20170717.csv"
VESSELS_FILENAME = "transshipment-vessels-v20170717.csv"

In [ ]:
transshipment_data_dir = os.path.join(os.getcwd(), TRANSSHIPMENT_DIR_FROM_CWD)

encounter_path = os.path.join(transshipment_data_dir, ENCOUNTER_EVENTS_FILENAME)
loitering_path = os.path.join(transshipment_data_dir, LOITERING_EVENTS_FILENAME)
transshipment_path = os.path.join(transshipment_data_dir, VESSELS_FILENAME)

encounter_df = pd.read_csv(encounter_path)
loitering_df = pd.read_csv(loitering_path)
transshipment_df = pd.read_csv(transshipment_path)


print("="*20 + " Encounter " + "="*20)
print("Shape: ",encounter_df.shape)
print("Example Row: ",encounter_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Loitering " + "="*20)
print("Shape: ",loitering_df.shape)
print("Example Row: ",loitering_df.head(1))
print("="*20 + "=========" + "="*20)

print()

print("="*20 + " Transshipment " + "="*20)
print("Shape: ",transshipment_df.shape)
print("Example Row: ",transshipment_df.head(1))
print("="*20 + "=========" + "="*20)

## Folium Maps

In [ ]:
map = folium.Map(location=[0, 0], zoom_start=2, tiles="CartoDB positron")

marker_cluster = MarkerCluster()

for ship in tqdm(encounter_df.iterrows(), desc="Generating Map...", total=encounter_df.shape[0]):

  lat, long = ship[1]["mean_latitude"], ship[1]["mean_longitude"]

  ship_name = transshipment_df[transshipment_df["mmsi"] == ship[1]["transshipment_vessel_mmsi"]]["shipname"].values
  if ship_name.size > 0:
    ship_name = ship_name[0]
  else:
    ship_name = "Unknown"

  folium.Marker(
      location=[lat, long],
      popup=ship_name,
      icon=folium.Icon(color="red", icon="ship")
  ).add_to(marker_cluster)

marker_cluster.add_to(map)

print("Map showing the locations of vessels transferring product onto another vessel")
map


# Plotting Global Marine Protected Areas (MPAs)

In [ ]:
import fiona
import geopandas as gpd
import plotly.express as px
import matplotlib.pyplot as plt

In [ ]:
MPA_GDB_PATH_FROM_CWD = "../data/raw/global_mpa/WDPA_WDOECM_Apr2026_Public_marine.gdb"

In [ ]:
mpa_gdb_path = os.path.join(os.getcwd(), MPA_GDB_PATH_FROM_CWD)

fiona.listlayers(mpa_gdb_path)

In [ ]:
# mpa_poly_gdf = gpd.read_file(mpa_gdb_path, layer="WDPA_WDOECM_poly_Apr2026_marine")
# mpa_poly_gdf.plot()

In [ ]:
mpa_point_gdf = gpd.read_file(mpa_gdb_path, layer="WDPA_WDOECM_point_Apr2026_marine")
# mpa_point_gdf.plot()

In [ ]:
mpa_location = mpa_point_gdf[['NAME_ENG', 'geometry']]

# extract lat and long

mpa_location['geometry'] = mpa_location['geometry'].explode(ignore_index=True)
mpa_location['lat'] = mpa_location['geometry'].y.astype(float)
mpa_location['long'] = mpa_location['geometry'].x.astype(float)
mpa_location.head()

# Feature Engineering

In [ ]:
loitering_loc = loitering_df[['transshipment_mmsi', 'starting_latitude', 'starting_longitude', 'ending_latitude', 'ending_longitude', 'total_event_duration', 'median_speed_knots', 'starting_timestamp']]

# Calculate the differences in lat and long from starting and ending positions
loitering_loc['diff_lat'] = loitering_loc['ending_latitude'] - loitering_loc['starting_latitude']
loitering_loc['diff_long'] = loitering_loc['ending_longitude'] - loitering_loc['starting_longitude']

# Determine if activity occured during night by correcting time (Local solar hour = UTC + longitude/15)
loitering_loc['starting_timestamp'] = pd.to_datetime(loitering_loc['starting_timestamp'])
loitering_loc['local_hour'] = (
    loitering_loc['starting_timestamp'].dt.hour +
    loitering_loc['starting_longitude'] / 15
) % 24

# Redefine is_night (True if between 6 PM and 6 AM)
loitering_loc['is_night_corrected'] = (
    (loitering_loc['local_hour'] >= 18) |
    (loitering_loc['local_hour'] < 6)
).astype(int)


loitering_loc['movement_mag'] = np.sqrt(
    loitering_loc['diff_lat']**2 + loitering_loc['diff_long']**2
)

#Calculate the distance to the nearest marine protected areas

loitering_nearest_mpa = loitering_loc.copy()

loitering_nearest_mpa['nearest_mpa_name'] = ""
loitering_nearest_mpa['nearest_mpa_lat'] = 0
loitering_nearest_mpa['nearest_mpa_long'] = 0
loitering_nearest_mpa['nearest_mpa_dist'] = 0


loit_coords = loitering_nearest_mpa[['starting_latitude', 'starting_longitude']].values
mpa_coords = mpa_location[['lat', 'long']].values


indices = [np.linalg.norm(mpa_coords - point, axis=1).argmin() for point in loit_coords]

nearest_mpas = mpa_location.iloc[indices]
loitering_nearest_mpa['nearest_mpa_name'] = nearest_mpas['NAME_ENG'].values
loitering_nearest_mpa['nearest_mpa_lat'] = nearest_mpas['lat'].values
loitering_nearest_mpa['nearest_mpa_long'] = nearest_mpas['long'].values

loitering_nearest_mpa['nearest_mpa_dist'] = np.sqrt(
    (loitering_nearest_mpa['starting_latitude'] - loitering_nearest_mpa['nearest_mpa_lat'])**2 +
    (loitering_nearest_mpa['starting_longitude'] - loitering_nearest_mpa['nearest_mpa_long'])**2
)

loitering_nearest_mpa.head()

# Visualizing loitering ships and their proximity to the nearest MPA

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Create the lines using a single trace for performance
lats = []
lons = []

for _, row in loitering_nearest_mpa.iterrows():
    lats.extend([row['starting_latitude'], row['nearest_mpa_lat'], None])
    lons.extend([row['starting_longitude'], row['nearest_mpa_long'], None])

fig.add_trace(go.Scattermapbox(
    mode="lines",
    lat=lats,
    lon=lons,
    line=dict(width=1, color="red"),
    opacity=0.3,
    hoverinfo='none'
))

fig.update_layout(mapbox_style="carto-positron", margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

# PCA Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px
import pandas as pd
import numpy as np

# --- 1. PREPARATION ---
features = ['total_event_duration', 'median_speed_knots', 'nearest_mpa_dist', 'movement_mag', 'is_night_corrected', 'starting_latitude', 'starting_longitude']
X = loitering_nearest_mpa[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- 2. PCA DIAGNOSTICS (Cumulative Variance) ---
# Fit PCA with all components to see the "Full Picture"
pca_full = PCA()
pca_full.fit(X_scaled)
cum_sum = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_sum) + 1), cum_sum, marker='o', linestyle='--', color='b')
plt.title('PCA Cumulative Explained Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.axhline(y=0.90, color='r', linestyle='-') # 90% threshold line
plt.grid()
plt.show()

# Now reduce to 2 components for the visual dashboard as planned
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
loitering_nearest_mpa['pca_1'] = X_pca[:, 0]
loitering_nearest_mpa['pca_2'] = X_pca[:, 1]

# --- 3. CLUSTERING DIAGNOSTICS (WCSS & Silhouette) ---
wcss = []
silhouette_avg = []
k_range = range(2, 10) # Silhouette requires at least 2 clusters

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_pca)

    # WCSS (Inertia)
    wcss.append(kmeans.inertia_)

    # Silhouette Score (Closer to 1 is better)
    # Sample data for silhoueete if too slow
    silhouette_avg.append(silhouette_score(X_pca, cluster_labels, sample_size=10000))
    # silhouette_avg.append(silhouette_score(X_pca, cluster_labels))

# Plotting Elbow and Silhouette side-by-side
fig, ax1 = plt.subplots(figsize=(12, 5))

# Elbow Plot (WCSS)
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('WCSS (Inertia)', color='tab:blue')
ax1.plot(k_range, wcss, marker='o', color='tab:blue', label='WCSS')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Silhouette Plot
ax2 = ax1.twinx()
ax2.set_ylabel('Silhouette Score', color='tab:red')
ax2.plot(k_range, silhouette_avg, marker='s', color='tab:red', label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title('Clustering Evaluation: Elbow Method & Silhouette Score')
plt.show()

# --- 4. FINAL CLUSTERING & VISUALIZATION ---
# Based on the plots above, choose the 'k' where Silhouette is high and WCSS begins to level off
optimal_k = 3
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
loitering_nearest_mpa['cluster'] = kmeans_final.fit_predict(X_pca)

# Final PCA Scatter Plot
fig = px.scatter(
    loitering_nearest_mpa,
    x='pca_1', y='pca_2',
    color='cluster',
    symbol='cluster',
    hover_data=features + ['nearest_mpa_name'],
    title=f"Vessel Behavioral Clusters (k={optimal_k}) in PCA-Reduced Space",
    color_discrete_sequence=px.colors.qualitative.Vivid,
    template="plotly_dark"
)
fig.show()

In [ ]:
#Print loadings for each PC
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=features
)
print(loadings)

In [ ]:
#Print cluster mean

cluster_means = loitering_nearest_mpa.groupby('cluster')[features].mean()
print(cluster_means)